# TMC-LM Training on Colab (T4 GPU)
Run all cells: `Runtime > Run all` | Need GPU: `Runtime > Change runtime type > T4 GPU`

Repo: `4r6ie/tmc-llm` (fork from McEmil1993) | Branch `main` | Dataset 1,000+ examples (multi-document + QA pairs)

In [ ]:
# 1. Clone repo (kung wala pa) - imong fork na 4r6ie/tmc-llm
!git clone https://github.com/4r6ie/tmc-llm.git
%cd tmc-llm
!git pull origin main
!ls -lh data/raw/tmc_sources/
# upstream (original) kung gusto nimo i-sync: !git remote add upstream https://github.com/McEmil1993/tmc-llm.git

In [ ]:
# 2. Check GPU (dapat T4)
!nvidia-smi
import torch; print('cuda:', torch.cuda.is_available(), '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 3. Install dependencies
!pip install -q -r requirements.txt
!pip install -q -e .
!python -m pip show transformers peft accelerate | grep -E "Name|Version"

In [ ]:
# 4. Build dataset (1000+ examples from multi-document sources + curated QA pairs)
!python -m tmc_llm.dataset_builder --source-dir data/raw/tmc_sources --output-dir data/processed
!cat data/processed/metadata.json
!cat data/processed/sources_manifest.json
!wc -l data/processed/train.jsonl data/processed/validation.jsonl

In [ ]:
# 5. Train LoRA - pinaka dugay ~25-40 mins sa T4 GPU (300 steps, configs/train_lora.yaml, v1.0)
!python -m tmc_llm.train_lora --config configs/train_lora.yaml
!ls -lh models/adapters/tmc-lm-tinyllama-lora-v1.0/

In [ ]:
# 6. Merge LoRA -> HF merged model (v1.0)
!python -m tmc_llm.merge_lora --base-model TinyLlama/TinyLlama-1.1B-Chat-v1.0 --adapter-dir models/adapters/tmc-lm-tinyllama-lora-v1.0 --output-dir models/merged/tmc-lm-tinyllama-v1.0
!ls -lh models/merged/tmc-lm-tinyllama-v1.0/ | head -20

In [ ]:
# 7. Convert to GGUF (need llama.cpp - clone if wala)
!test -d external/llama.cpp || git clone https://github.com/ggml-org/llama.cpp external/llama.cpp
!cmake -B external/llama.cpp/build -DCMAKE_BUILD_TYPE=Release -S external/llama.cpp && cmake --build external/llama.cpp/build --config Release -j$(nproc)
!python external/llama.cpp/convert_hf_to_gguf.py models/merged/tmc-lm-tinyllama-v1.0 --outfile models/gguf/tmc-lm-tinyllama-f16.gguf --outtype f16
!external/llama.cpp/build/bin/llama-quantize models/gguf/tmc-lm-tinyllama-f16.gguf models/gguf/tmc-lm-tinyllama-q4_k_m.gguf Q4_K_M
!python -m tmc_llm.gguf_check --path models/gguf/tmc-lm-tinyllama-q4_k_m.gguf
!ls -lh models/gguf/

In [ ]:
# 8. Save to Google Drive (para dili mawala)
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p "/content/drive/MyDrive/tmc-llm-models"
!cp -r models/adapters/tmc-lm-tinyllama-lora-v1.0 "/content/drive/MyDrive/tmc-llm-models/"
!cp -r models/merged/tmc-lm-tinyllama-v1.0 "/content/drive/MyDrive/tmc-llm-models/"
!cp -r models/gguf "/content/drive/MyDrive/tmc-llm-models/"
!cp -r data/processed "/content/drive/MyDrive/tmc-llm-models/"
!ls -lh "/content/drive/MyDrive/tmc-llm-models/"

In [ ]:
# 9. Quick inference test using llama.cpp CLI (GGUF quality check)
# Paspas lang nga test aron makita dayon ang quality sa training
# Questions: vision, programs, admission requirements, mission, registrar office
!external/llama.cpp/build/bin/llama-cli -m models/gguf/tmc-lm-tinyllama-q4_k_m.gguf \
  -p "You are TMC-LM, an offline assistant for Trinidad Municipal College. Answer using only the provided official TMC knowledge. If the source does not contain the answer, say that the available TMC source does not contain it.\n\nUser: What is TMC's vision?\nAssistant:" \
  -n 128 -t 4 --temp 0.2 --top-p 0.9 --repeat-penalty 1.12
print("-----")
!external/llama.cpp/build/bin/llama-cli -m models/gguf/tmc-lm-tinyllama-q4_k_m.gguf \
  -p "User: What programs are offered at TMC?\nAssistant:" \
  -n 128 -t 4 --temp 0.2 --top-p 0.9 --repeat-penalty 1.12

In [ ]:
# 9b. Inference test with merged HF model (5 standard evaluation questions)
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model = AutoModelForCausalLM.from_pretrained(
    "models/merged/tmc-lm-tinyllama-v1.0",
    torch_dtype=torch.float16,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained("models/merged/tmc-lm-tinyllama-v1.0")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

EVAL_QUESTIONS = [
    "What is TMC's vision?",
    "What programs are offered at Trinidad Municipal College?",
    "What are the admission requirements at TMC?",
    "What is TMC's mission?",
    "What does the Registrar Office manage?",
]

for question in EVAL_QUESTIONS:
    messages = [
        {"role": "system", "content": "You are TMC-LM, an offline assistant for Trinidad Municipal College. Answer using only the provided official TMC knowledge. If the source does not contain the answer, say that the available TMC source does not contain it."},
        {"role": "user", "content": question},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.2,
        top_p=0.9,
        repetition_penalty=1.12,
        do_sample=True,
    )
    answer = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    print(f"Q: {question}")
    print(f"A: {answer.strip()}\n")
    print("=" * 60)

In [ ]:
# 9c. Ollama smoke test (kung naka-install ang llama.cpp server/ollama)
# Alternative to Cell 9: direct llama.cpp CLI test gamit ang GGUF
!external/llama.cpp/build/bin/llama-cli -m models/gguf/tmc-lm-tinyllama-q4_k_m.gguf \
  -p "User: What is the vision of TMC?\nAssistant:" \
  -n 256 -t 4 --temp 0.2 --top-p 0.9 --repeat-penalty 1.12

### Next: Test sa Colab
`!python -c "from transformers import AutoModelForCausalLM; print('ok')"`

### Pull back to VSCode
Sa VSCode: `git pull origin main` kung naa kay gi-update sa Colab (kung gi-push).